In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = pd.read_csv('data/data.csv')


# # extracting a weekday
# data_05['Datetime'] = pd.to_datetime({
#     'year': data_05['Year'],
#     'month': data_05['Month'],
#     'day': data_05['Day']})

# data_05['Weekday'] = data_05['Datetime'].dt.weekday

# data_05.drop(columns=['Datetime'], inplace=True)

# data_09['Datetime'] = pd.to_datetime({
#     'year': data_09['Year'],
#     'month': data_09['Month'],
#     'day': data_09['Day']})

# data_09['Weekday'] = data_09['Datetime'].dt.weekday
# data_09.drop(columns=['Datetime'], inplace=True)



In [7]:
# building function to get the basic info about the dataset

def visual_data(df, hist_plot=True, corr_plot=True, multicollinear=True, corr_threshold=0.80):
    # basic info
    print("Data Information:")
    df.info()
    
    # first 10 rows
    print("\nFirst 10 rows:")
    print(df[:10])
    
    # duplicates
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = (duplicate_count / len(df)) * 100
    print(f"\nDuplicate Rows: {duplicate_count} ({duplicate_percentage:.2f}% of total rows)")

    # missing values
    missing_percentage = (df.isna().mean() * 100).round(2)
    total_missing = missing_percentage.sum()
    if total_missing > 0:
        print("\nMissing Values (%):")
        print(missing_percentage[missing_percentage > 0].sort_values(ascending=False))
    else:
        print(f"\nMissing Values: 0.00%")
    
    # descriptive statistics for numeric columns
    print("\nDescriptive Statistics (Numerical):")
    print(df.describe().T.round(2))

    # descriptive statistics for categorical columns
    categorical_cols = df.select_dtypes(include=['object', 'category'])
    print("\nDescriptive Statistics (Categorical):")
    print(categorical_cols.describe().T)

    # histogram
    if hist_plot:
        df.hist(bins=20, color='blue', edgecolor='black', 
        grid=False,figsize=(12, 8))
        plt.tight_layout()
        plt.show()
    
    # correlation heatmap
    if corr_plot:
        numeric_cols = df.select_dtypes(include=['number'])
        plt.figure(figsize=(8, 6))
        corr_matrix = numeric_cols.corr()
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', 
        fmt='.2f', linewidths=0.5)
        plt.title('Correlation Matrix')
        plt.show()

    # multicollinearity
    if multicollinear and not numeric_cols.empty:
        print("\nChecking for Multicollinearity (Correlation Threshold > {:.2f}):".format(corr_threshold))
        high_corr_pairs = []
        corr_matrix = numeric_cols.corr().abs()  # Calculate correlation matrix for numeric columns only
        for i in range(len(corr_matrix.columns)):
            for j in range(i):
                if corr_matrix.iloc[i, j] > corr_threshold:
                    high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
        
        if high_corr_pairs:
            print("High correlation pairs found:")
            for pair in high_corr_pairs:
                print(f" - {pair[0]} and {pair[1]}: Correlation = {pair[2]:.2f}")
        else:
            print("No highly correlated pairs found.")

    return df

In [ ]:
def process_observations(df):
    """
    Processes the DataFrame to select the best observation for each group.
    
    The function performs the following steps:
      1. Creates a temporary column 'Temp_ID' by combining the columns 'ID', 'Month', 'Day', and 'Time'.
      2. Groups the data by 'Temp_ID', 'Start (s)', and 'End (s)'.
      3. Within each group, applies the selection logic:
            - If all observations have Habitant == 1, pick the row with the highest Confidence.
            - If all observations have Habitant == 0, pick the row with the highest Confidence.
            - If there is a mix, pick the row with the highest Confidence among those with Habitant == 1.
      4. Drops the temporary 'Temp_ID' column from the final output.
      
    Parameters:
        df (pd.DataFrame): The input DataFrame with the required columns.
        
    Returns:
        pd.DataFrame: The processed DataFrame with the best observation per group.
    """
    
    # Create the temporary column 'Temp_ID'
    df['Temp_ID'] = df['ID'].astype(str) + df['Month'].astype(str) + df['Day'].astype(str) + df['Time'].astype(str)
    
    # Function to select the best observation per group based on the conditions
    def choose_best_row(group):
        if all(group['Habitant'] == 1):
            return group.loc[group['Confidence'].idxmax()]
        elif all(group['Habitant'] == 0):
            return group.loc[group['Confidence'].idxmax()]
        else:
            subset = group[group['Habitant'] == 1]
            if not subset.empty:
                return subset.loc[subset['Confidence'].idxmax()]
            else:
                return group.loc[group['Confidence'].idxmax()]
    
    # Group by Temp_ID, Start (s), and End (s) and apply the selection function
    selected_rows = (
        df.groupby(['Temp_ID', 'Start (s)', 'End (s)'], as_index=False)
          .apply(choose_best_row)
          .reset_index(drop=True)
    )
    
    # Drop the temporary Temp_ID column from the final output
    selected_rows = selected_rows.drop(columns=['Temp_ID'])
    
    return selected_rows

In [ ]:
# a function that defines who appears before and after Hume's Warbler ( target specie)

def birds_around(df, name='phylloscopus humei', window=12):
  
    # Filter for events matching the target scientific name
    events = df[df['Scientific name'] == name]
    output_rows = []
    
    for idx, event in events.iterrows():
        # Define the time windows:
        window_before_start = event['Start (s)'] - window
        window_before_end   = event['Start (s)']
        
        window_after_start = event['End (s)']
        window_after_end   = event['End (s)'] + window

        # Select birds in the "before" window
        birds_before = df[
            (df['Start (s)'] >= window_before_start) &
            (df['End (s)'] <= window_before_end)
        ]
        
        # Add window info for birds before the event
        for _, bird in birds_before.iterrows():
            row = bird.to_dict()
            row['Event_Start'] = event['Start (s)']
            row['Event_End'] = event['End (s)']
            row['Window'] = 'Before'
            output_rows.append(row)
        
        # Select birds in the "after" window
        birds_after = df[
            (df['Start (s)'] >= window_after_start) &
            (df['End (s)'] <= window_after_end)
        ]
        
        # Add window info for birds after the event
        for _, bird in birds_after.iterrows():
            row = bird.to_dict()
            row['Event_Start'] = event['Start (s)']
            row['Event_End'] = event['End (s)']
            row['Window'] = 'After'
            output_rows.append(row)
    
    result_df = pd.DataFrame(output_rows)
    return result_df

In [ ]:
# merging cocontiguous intervals for rows with the same 'Scientific name' 

def merge_contiguous_intervals(df):

    # Initialize the list to collect merged rows
    merged_rows = []
    
    # Start with the first row as the current interval
    current = df.iloc[0].copy()
    
    # Iterate over the DataFrame from the second row onward
    for idx in range(1, len(df)):
        row = df.iloc[idx]
        
        # If the same species and the intervals are contiguous, merge them
        if (row['Scientific name'] == current['Scientific name'] and 
            row['Start (s)'] == current['End (s)']):
            # Extend the current interval's end time
            current['End (s)'] = row['End (s)']
        else:
            # Append the current interval and start a new one
            merged_rows.append(current)
            current = row.copy()
    

    merged_rows.append(current)
    
    merged_df = pd.DataFrame(merged_rows)
    
    # Remove rows with a 3-second interval (i.e., where duration equals 3 seconds)
    merged_df = merged_df[(merged_df['End (s)'] - merged_df['Start (s)']) != 3]
    
    # Reset the index of the final DataFrame
    merged_df = merged_df.reset_index(drop=True)
    
    return merged_df


